# 04 · Data Quality & Reconciliation
Runs after every load. Results are written to `ops_dq_results` and surfaced on the
**Data Health** page of the Power BI report. A failed *critical* check raises an exception so the
Data Factory pipeline stops and the failure alert fires.

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime
results, run_ts = [], datetime.utcnow()

def check(name, table, severity, failed_rows, total_rows=None):
    status = "PASS" if failed_rows == 0 else "FAIL"
    results.append((run_ts, name, table, severity, int(failed_rows), int(total_rows or 0), status))
    print(f"{'✅' if status=='PASS' else '❌'} [{severity}] {name}: {failed_rows} failed")

In [ ]:
emp = spark.table("silver_employee")
n = emp.count()

# Completeness
check("Employee ID not null",          "silver_employee", "CRITICAL", emp.filter("employee_id IS NULL").count(), n)
check("Hire date not null",            "silver_employee", "CRITICAL", emp.filter("hire_date IS NULL").count(), n)
# Uniqueness
check("Employee ID unique",            "silver_employee", "CRITICAL", n - emp.select("employee_id").distinct().count(), n)
# Validity
check("Termination after hire",        "silver_employee", "CRITICAL", emp.filter("termination_date < hire_date").count(), n)
check("Gender standardised",           "silver_employee", "WARNING",
      emp.filter(~F.col("gender").isin("Male", "Female", "Non-binary", "Not Disclosed")).count(), n)
# Referential integrity
for fk, dim in [("department_id", "dim_department"), ("location_id", "dim_location"), ("grade_id", "dim_grade")]:
    check(f"{fk} exists in {dim}", "dim_employee", "CRITICAL",
          spark.table("dim_employee").join(spark.table(dim), fk, "left_anti").count(), n)
for fact in ["fact_payroll", "fact_training", "fact_leave", "fact_performance", "fact_exit"]:
    check(f"{fact} employee exists", fact, "WARNING",
          spark.table(fact).join(spark.table("dim_employee"), "employee_id", "left_anti").count())
# Business rules
check("Performance rating 1-5", "fact_performance", "CRITICAL",
      spark.table("fact_performance").filter("performance_rating NOT BETWEEN 1 AND 5").count())
check("Net pay not negative",   "fact_payroll", "CRITICAL", spark.table("fact_payroll").filter("net_pay < 0").count())
# Reconciliation: Bronze vs Silver (allowing for duplicates + quarantine)
b = spark.table("bronze_hris_employees").select("employee_id").distinct().count()
q = spark.table("quarantine_employee").count()
check("Employee reconciliation (bronze = silver + quarantine)", "silver_employee", "CRITICAL", abs(b - (n + q)), b)

In [ ]:
res = spark.createDataFrame(results,
    "run_ts timestamp, check_name string, table_name string, severity string, failed_rows long, total_rows long, status string")
res.write.format("delta").mode("append").saveAsTable("ops_dq_results")
display(res)

critical_fail = res.filter("severity = 'CRITICAL' AND status = 'FAIL'").count()
if critical_fail:
    raise Exception(f"{critical_fail} critical data-quality check(s) failed – pipeline stopped.")